# Run v3 on the LIARArg test set (the 83% empty finding)Parses the LIARArg test rows with the v3 adapter. Hits 83% empty rate with fragmentary outputs. This is the failure mode cited in Section 5.3 of the paper as the motivation for adding silver labels and Chain of Thought supervision in v4.Recommended to rename this file to `v3_liar_parse.ipynb`.

In [ ]:
%%bash
cd ~/argument-aware-rag && source .venv/bin/activate
mkdir -p phase2_data_liar eval_logs
nohup python3 -u <<'PY' > eval_logs/phase2beta_v3_liar_parse.log 2>&1 &
import sys, json, time, os
sys.path.insert(0, '.')
from pathlib import Path
from src.phase2.config import load_phase2_config
from src.phase2.student import build_student

cfg = load_phase2_config('configs/phase2_beta_qwen1.5b_lora_v3.yaml')
student = build_student(cfg.student)
student.load(cfg.student_output_dir)
print(f"[parse-v3] loaded student.  use_cache={student._model.config.use_cache}",
      flush=True)

OUT = Path('phase2_data_liar/parser_preds_phase2beta_v3.jsonl')
OUT.parent.mkdir(parents=True, exist_ok=True)

# Resume support
done_ids = set()
if OUT.exists():
    with open(OUT) as f:
        for line in f:
            try: done_ids.add(int(json.loads(line)['row_id']))
            except: pass
print(f"[parse-v3] resuming — {len(done_ids)} predictions on disk", flush=True)

with open('data/test.jsonl') as f:
    all_rows = [json.loads(l) for l in f if l.strip()]
todo_rows = [r for r in all_rows if int(r['id']) not in done_ids]
print(f"[parse-v3] {len(todo_rows)} rows to do (of {len(all_rows)} total)",
      flush=True)

BATCH = 4
TRUNCATE_CHARS = 1500   # LIARArg full_text often > 4000 chars; truncate to
                        # something closer to AbstRCT-style length where v3
                        # learned to perform

def get_text(row):
    text = row.get('full_text') or row.get('summary') or row.get('statement', '')
    return text[:TRUNCATE_CHARS]

t0 = time.time()
with open(OUT, 'a') as fout:
    for i in range(0, len(todo_rows), BATCH):
        batch = todo_rows[i:i+BATCH]
        texts = [get_text(r) for r in batch]
        try:
            batch_preds = student.predict_batch(texts)
            batch_preds = [p[0] if isinstance(p, tuple) else p
                           for p in batch_preds]
        except Exception as e:
            print(f"  ERROR at batch {i}: {e}", flush=True)
            batch_preds = [{'claim_components':[],'premise_components':[],
                            'citation_components':[],'relations':[]}
                           for _ in batch]
        for row, pred in zip(batch, batch_preds):
            fout.write(json.dumps({
                'row_id': int(row['id']),
                'prediction': pred,
                'reasoning': '',
            }, ensure_ascii=False) + '\n')
        fout.flush(); os.fsync(fout.fileno())

        done = min(i+BATCH, len(todo_rows))
        elapsed = time.time() - t0
        rate = done / max(elapsed, 1e-6)
        eta = (len(todo_rows) - done) / max(rate, 1e-6) / 60
        if (i // BATCH) % 5 == 0:
            print(f"  {done}/{len(todo_rows)} ({elapsed:.0f}s, "
                  f"{rate:.2f}/s, ETA {eta:.1f} min)", flush=True)

# Final stats
total, empty = 0, 0
with open(OUT) as f:
    for line in f:
        rec = json.loads(line)
        total += 1
        p = rec['prediction']
        if (len(p['claim_components'])==0 and len(p['premise_components'])==0):
            empty += 1
print(f"\n[parse-v3] DONE. {total} rows | empty: {empty} "
      f"({100*empty/total:.1f}%) | real: {total-empty}", flush=True)
PY
echo "PID: $!"
sleep 10
tail -15 eval_logs/phase2beta_v3_liar_parse.log

In [ ]:
%%bash
pkill -f "parse-v3\|phase2beta_v3" 2>/dev/null && echo "killed" || echo "no process found"